# Generate Skewed Data 

## Import Modules

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.functions import udf
from pyspark.sql.types import *

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import random
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

In [2]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "8g")
    .config("spark.driver.cores", "4")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/27 21:23:23 WARN Utils: Your hostname, codebase-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/06/27 21:23:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/27 21:23:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Setting Up Transactions Data

In [26]:
customers_file = "/Users/codebase/Documents/codebase/Courses/spark-experiments/data/data_skew/customers.parquet"
txns_file = "/Users/codebase/Documents/codebase/Courses/spark-experiments/data/data_skew/transactions.parquet"

In [27]:
!pwd

/Users/codebase/Documents/codebase/Courses/spark-experiments/spark/1_data_skew


In [28]:
df_raw_txns = spark.read.parquet(txns_file, header=True)

In [29]:
df_raw_txns.printSchema()
df_raw_txns.show(3, False)

root
 |-- cust_id: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- city: string (nullable = true)

+----------+----------+----------+---------------+----------+----+-----+---+-------------+-----+--------+
|cust_id   |start_date|end_date  |txn_id         |date      |year|month|day|expense_type |amt  |city    |
+----------+----------+----------+---------------+----------+----+-----+---+-------------+-----+--------+
|C0YDPQWPBJ|2010-07-01|2018-12-01|TZ5SMKZY9S03OQJ|2018-10-07|2018|10   |7  |Entertainment|10.42|boston  |
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYIAPPNU066CJ5R|2016-03-27|2016|3    |27 |Motor/Travel |44.34|portland|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TETSXIK4BLXH

In [30]:
df_txns = (
    df_raw_txns.withColumnRenamed("CUST_ID", "cust_id")
    .withColumnRenamed("START_DATE", "start_date")
    .withColumnRenamed("END_DATE", "end_date")
    .withColumnRenamed("TRANS_ID", "txn_id")
    .withColumnRenamed("DATE", "date")
    .withColumnRenamed("YEAR", "year")
    .withColumnRenamed("MONTH", "month")
    .withColumnRenamed("DAY", "day")
    .withColumnRenamed("EXP_TYPE", "expense_type")
    .withColumnRenamed("AMOUNT", "amt")
)

In [31]:
df_txns.printSchema()
df_txns.show(3, False)

root
 |-- cust_id: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- city: string (nullable = true)

+----------+----------+----------+---------------+----------+----+-----+---+-------------+-----+--------+
|cust_id   |start_date|end_date  |txn_id         |date      |year|month|day|expense_type |amt  |city    |
+----------+----------+----------+---------------+----------+----+-----+---+-------------+-----+--------+
|C0YDPQWPBJ|2010-07-01|2018-12-01|TZ5SMKZY9S03OQJ|2018-10-07|2018|10   |7  |Entertainment|10.42|boston  |
|C0YDPQWPBJ|2010-07-01|2018-12-01|TYIAPPNU066CJ5R|2016-03-27|2016|3    |27 |Motor/Travel |44.34|portland|
|C0YDPQWPBJ|2010-07-01|2018-12-01|TETSXIK4BLXH

# Setting Up Customer Data

In [32]:
df_customer_det = spark.read.parquet(customers_file, header=True)

In [33]:
df_customer_det.printSchema()
df_customer_det.show(3, False)

root
 |-- cust_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthday: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- city: string (nullable = true)

+----------+------------+---+------+----------+-----+-------+
|cust_id   |name        |age|gender|birthday  |zip  |city   |
+----------+------------+---+------+----------+-----+-------+
|C007YEYTX9|Aaron Abbott|34 |Female|7/13/1991 |97823|boston |
|C00B971T1J|Aaron Austin|37 |Female|12/16/2004|30332|chicago|
|C00WRSJF1Q|Aaron Barnes|29 |Female|3/11/1977 |23451|denver |
+----------+------------+---+------+----------+-----+-------+
only showing top 3 rows


In [34]:
df_top5k_customers = (
    df_txns
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("distinct_txns"))
    .orderBy(F.desc("distinct_txns"))
    .limit(5000)
    .withColumn("row_id", F.row_number().over(Window.orderBy("cust_id")))
)

In [35]:
df_customer_det = df_customer_det.withColumn("row_id", F.row_number().over(Window.orderBy("name")))
df_customer_identity = df_top5k_customers.join(df_customer_det, "row_id").drop("row_id")

In [36]:
df_customer_identity.printSchema()
df_customer_identity.show(5, False)
df_customer_identity.select("cust_id").distinct().count()
df_customer_identity.count()

root
 |-- cust_id: string (nullable = true)
 |-- distinct_txns: long (nullable = false)
 |-- cust_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthday: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- city: string (nullable = true)



{"ts": "2026-06-27 21:26:54.479", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `cust_id` is ambiguous, could be: [`cust_id`, `cust_id`]. SQLSTATE: 42704", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o164.select.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `cust_id` is ambiguous, could be: [`cust_id`, `cust_id`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2232)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:356)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHel

+----------+-------------+----------+-------------+---+------+----------+-----+-----------+
|cust_id   |distinct_txns|cust_id   |name         |age|gender|birthday  |zip  |city       |
+----------+-------------+----------+-------------+---+------+----------+-----+-----------+
|C007YEYTX9|7445         |C007YEYTX9|Aaron Abbott |34 |Female|7/13/1991 |97823|boston     |
|C00B971T1J|7532         |C00B971T1J|Aaron Austin |37 |Female|12/16/2004|30332|chicago    |
|C00WRSJF1Q|7777         |C00WRSJF1Q|Aaron Barnes |29 |Female|3/11/1977 |23451|denver     |
|C01AZWQMF3|7548         |C01AZWQMF3|Aaron Barrett|31 |Male  |7/9/1998  |46613|los_angeles|
|C01BKUFRHA|7401         |C01BKUFRHA|Aaron Becker |54 |Male  |11/24/1979|40284|san_diego  |
+----------+-------------+----------+-------------+---+------+----------+-----+-----------+
only showing top 5 rows


AnalysisException: [AMBIGUOUS_REFERENCE] Reference `cust_id` is ambiguous, could be: [`cust_id`, `cust_id`]. SQLSTATE: 42704

In [ ]:
def assign_city():
    cities = [
        'san_francisco', 'new_york', 'chicago', 'philadelphia', 
        'boston', 'seattle', 'san_diego', 'los_angeles', 
        'denver', 'portland'
    ]
    return random.choice(cities)

assign_city_udf = udf(assign_city, StringType())

In [14]:
df_customer_identity = (
    df_customer_identity.withColumn(
        "city",
        assign_city_udf()
    )
)

In [15]:
df_customer_identity.show(10, False)

+----------+-------------+-------------+---+------+----------+-----+------------+
|cust_id   |distinct_txns|name         |age|gender|birthday  |zip  |city        |
+----------+-------------+-------------+---+------+----------+-----+------------+
|C007YEYTX9|7445         |Aaron Abbott |34 |Female|7/13/1991 |97823|denver      |
|C00B971T1J|7532         |Aaron Austin |37 |Female|12/16/2004|30332|portland    |
|C00WRSJF1Q|7777         |Aaron Barnes |29 |Female|3/11/1977 |23451|philadelphia|
|C01AZWQMF3|7548         |Aaron Barrett|31 |Male  |7/9/1998  |46613|philadelphia|
|C01BKUFRHA|7401         |Aaron Becker |54 |Male  |11/24/1979|40284|boston      |
|C01RGUNJV9|8280         |Aaron Bell   |24 |Female|8/16/1968 |86331|los_angeles |
|C01USDV4EE|7177         |Aaron Blair  |35 |Female|9/9/1974  |80078|philadelphia|
|C01WMZQ7PN|8617         |Aaron Brady  |51 |Female|8/20/1994 |52204|portland    |
|C021567NJZ|7260         |Aaron Briggs |57 |Male  |3/10/1990 |22008|new_york    |
|C023M6MKR3|7210

# Write Customer Data

In [16]:
(
    df_customer_identity
    .drop("distinct_txns")
    .write
    .mode("overwrite")
    .parquet("../../data/data_skew/customers.parquet")
)

# Write Skewed Transaction Data

In [17]:
# df_top10k_customers.filter(F.col("distinct_txns") >= 7000).distinct().count()

In [18]:
df_transactions = df_txns.join(
    df_top5k_customers,
    on="cust_id",
    how="inner"
).withColumn(
    "cust_id", 
    F.when(
        F.col("distinct_txns") >= 8000, F.lit("C0YDPQWPBJ")
    ).otherwise(F.col("cust_id"))
)

In [19]:
df_transactions = (
    df_transactions.withColumn(
        "city",
        assign_city_udf()
    )
)

In [20]:
df_transactions.groupBy("cust_id").count().orderBy(F.desc("count")).show(5, False)
df_transactions.cache()

+----------+--------+
|cust_id   |count   |
+----------+--------+
|C0YDPQWPBJ|17539732|
|CBW3FMEAU7|7999    |
|C3KUDEN3KO|7999    |
|C89FCEGPJP|7999    |
|CHNFNR89ZV|7998    |
+----------+--------+
only showing top 5 rows



DataFrame[cust_id: string, start_date: string, end_date: string, txn_id: string, date: string, year: string, month: string, day: string, expense_type: string, amt: string, distinct_txns: bigint, row_id: int, city: string]

In [21]:
df_transactions.printSchema()

root
 |-- cust_id: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- txn_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- distinct_txns: long (nullable = false)
 |-- row_id: integer (nullable = true)
 |-- city: string (nullable = true)



In [22]:
df_transactions.groupBy("cust_id", "city").count().orderBy(F.desc("count")).show(20, False)

+----------+-------------+-------+
|cust_id   |city         |count  |
+----------+-------------+-------+
|C0YDPQWPBJ|portland     |1756379|
|C0YDPQWPBJ|los_angeles  |1755910|
|C0YDPQWPBJ|denver       |1755398|
|C0YDPQWPBJ|san_francisco|1754952|
|C0YDPQWPBJ|seattle      |1754184|
|C0YDPQWPBJ|chicago      |1753398|
|C0YDPQWPBJ|boston       |1752906|
|C0YDPQWPBJ|san_diego    |1752767|
|C0YDPQWPBJ|philadelphia |1752140|
|C0YDPQWPBJ|new_york     |1751698|
|C4XLI291DF|denver       |877    |
|COYZEFEC9N|portland     |876    |
|CLU2H1C3GZ|portland     |874    |
|CNT7TK3O4N|new_york     |873    |
|CJJ0NUQIUD|chicago      |873    |
|C41YIKBMNX|los_angeles  |870    |
|CDSOKODPKL|chicago      |867    |
|CMMS3KUZ6S|philadelphia |867    |
|C3S3XFH3L3|portland     |866    |
|CZVAF6O8HI|san_francisco|865    |
+----------+-------------+-------+
only showing top 20 rows



In [23]:
# Checks to validate if data is sane

# df_transactions.select("cust_id").distinct().count()
# df_transactions.select("txn_id").distinct().count()
# df_transactions.count()

In [24]:
(
    df_transactions
    .drop("distinct_txns", "row_id")
    .write
    .mode("overwrite")
    .parquet("../../data/data_skew/transactions.parquet")
)

In [25]:
# df_transactions_test = spark.read.parquet("../../data/data_skew/transactions.parquet")

In [26]:
# Checks to validate if data is sane

# (
#     df_transactions_test
#     .groupBy("cust_id")
#     .agg(F.countDistinct("txn_id").alias("ct"))
#     .orderBy(F.desc("ct"))
#     .show(20, False)
# )

In [27]:
spark.stop()